# 📌 00_setup_models.ipynb — Model Registry & Offline Setup (v2: Sentiment-5)

本 notebook 负责：
- 下载并固化模型到本地 `models/`
- 锁定 revision（生成 lockfile）
- 生成 manifest（可审计）
- 跑离线 smoke tests（证明断网可推理）

⚠️ 第一次运行需要联网；完成后会自动做离线验证。

In [1]:
# Cell 1 — Bootstrap：项目根目录 + 生成/读取 registry + 预先设置 HF_ENDPOINT
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import yaml  # PyYAML

# --------------------------
# 0) Helper: find project root
# --------------------------
def find_project_root(start: Path | None = None) -> Path:
    """
    Try to find repo root by searching parent dirs for common markers.
    Falls back to current working directory.
    """
    start = (start or Path.cwd()).resolve()
    markers = [
        ".git",
        "pyproject.toml",
        "requirements.txt",
        "tree.txt",
        "configs",
        "models",
        "src",
        "docs",
    ]
    for p in [start, *start.parents]:
        if any((p / m).exists() for m in markers):
            return p
    return start

PROJECT_ROOT = find_project_root()
print("PROJECT_ROOT =", PROJECT_ROOT)

CONFIG_DIR = PROJECT_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

REGISTRY_PATH = CONFIG_DIR / "model_registry.yaml"
LOCK_PATH = CONFIG_DIR / "model_registry.lock.yaml"

# --------------------------
# 1) Create default registry if missing
# --------------------------
DEFAULT_REGISTRY = {
    # 留空表示使用默认 huggingface.co；如你在国内，可填 "https://hf-mirror.com"
    # 也可在系统环境变量提前设 HF_ENDPOINT，registry 里就不用写
    "hf_endpoint": "https://hf-mirror.com",
    "runtime_profile": "cpu_int8",  # cpu_int8 | xpu | openvino
    "include_optional_vad": False,  # 可选增强包
    "models": [
        {
            "model_key": "sentiment5_roberta_topic",
            "repo_id": "cardiffnlp/twitter-roberta-base-topic-sentiment-latest",
            "revision": "auto",  # auto=运行时解析成具体 commit sha 并写入 lockfile
            "local_dir": "models/sentiment5_roberta_topic",
            "task": "sentiment5",
        },
        {
            "model_key": "embedding_all-MiniLM-L6-v2",
            "repo_id": "sentence-transformers/all-MiniLM-L6-v2",
            "revision": "auto",
            "local_dir": "models/embedding_all-MiniLM-L6-v2",
            "task": "embedding",
        },
        {
            "model_key": "summarizer_distilbart-cnn-12-6",
            "repo_id": "sshleifer/distilbart-cnn-12-6",
            "revision": "auto",
            "local_dir": "models/summarizer_distilbart-cnn-12-6",
            "task": "summarization",
        },
        {
            "model_key": "nli_deberta_mnli_fever_anli",
            "repo_id": "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
            "revision": "auto",
            "local_dir": "models/nli_deberta_mnli_fever_anli",
            "task": "nli",
        },

        # Optional: VAD regression
        {
            "model_key": "vad_bert_optional",
            "repo_id": "RobroKools/vad-bert",
            "revision": "auto",
            "local_dir": "models/vad-bert",
            "task": "vad",
        },
    ],
}

if not REGISTRY_PATH.exists():
    with open(REGISTRY_PATH, "w", encoding="utf-8") as f:
        yaml.safe_dump(DEFAULT_REGISTRY, f, sort_keys=False, allow_unicode=True)
    print(f"✅ Created default registry: {REGISTRY_PATH}")
else:
    print(f"✅ Found registry: {REGISTRY_PATH}")

# --------------------------
# 2) Load registry and set env BEFORE importing huggingface_hub
# --------------------------
with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
    REGISTRY = yaml.safe_load(f)

hf_endpoint = (REGISTRY.get("hf_endpoint") or "").strip()
if hf_endpoint and not os.environ.get("HF_ENDPOINT"):
    os.environ["HF_ENDPOINT"] = hf_endpoint
    print("✅ HF_ENDPOINT set from registry:", hf_endpoint)
else:
    print("HF_ENDPOINT =", os.environ.get("HF_ENDPOINT", "(default)"))

# 推荐：关闭遥测
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

print("Bootstrap done.")


PROJECT_ROOT = D:\coding\projects\Python\Youtube Comment Sentiment Analysis
✅ Found registry: D:\coding\projects\Python\Youtube Comment Sentiment Analysis\configs\model_registry.yaml
✅ HF_ENDPOINT set from registry: https://hf-mirror.com
Bootstrap done.


In [2]:
# Cell 2 — 依赖检查 + 版本信息
# Optional: If you are missing packages, install them (uncomment):
# !pip install -U "transformers>=4.40" "huggingface_hub>=0.23" "sentence-transformers>=2.6" "pyyaml" "tqdm"

import platform
import time

try:
    import torch
    import transformers
    import huggingface_hub
    from huggingface_hub import HfApi
    from huggingface_hub import snapshot_download
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
    from sentence_transformers import SentenceTransformer
except Exception as e:
    raise RuntimeError(
        "Missing required packages. Please install:\n"
        "  pip install -U transformers huggingface_hub sentence-transformers pyyaml\n"
        f"Original error: {e}"
    )

print("Python:", sys.version)
print("Platform:", platform.platform())
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)


Python: 3.12.7 (tags/v3.12.7:0b05ead, Oct  1 2024, 03:06:41) [MSC v.1941 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
torch: 2.10.0+cpu
transformers: 4.57.6
huggingface_hub: 0.36.0


In [3]:
# Cell 3 — 路径初始化 + 读取旧 manifest（支持断点/离线复跑）
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
LOGS_DIR = PROJECT_ROOT / "logs"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = MODELS_DIR / "manifest.json"
SMOKE_REPORT_PATH = REPORTS_DIR / "00_smoke_test.json"

# Load existing manifest if any (helps skip network calls)
existing_manifest: Dict[str, Any] = {}
existing_by_key: Dict[str, Any] = {}

if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        existing_manifest = json.load(f)
    for entry in existing_manifest.get("models", []):
        existing_by_key[entry["model_key"]] = entry
    print(f"✅ Loaded existing manifest: {MANIFEST_PATH}")
else:
    print("No existing manifest found (first run).")

runtime_profile = (REGISTRY.get("runtime_profile") or "cpu_int8").strip()
include_optional_vad = bool(REGISTRY.get("include_optional_vad", False))

print("runtime_profile =", runtime_profile)
print("include_optional_vad =", include_optional_vad)


✅ Loaded existing manifest: D:\coding\projects\Python\Youtube Comment Sentiment Analysis\models\manifest.json
runtime_profile = cpu_int8
include_optional_vad = False


In [4]:
# Cell 4 — 工具函数：revision 解析、下载、manifest/lock 生成
import math
import hashlib

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def normalize_label(s: str) -> str:
    s = (s or "").strip().lower()
    s = s.replace("_", " ")
    s = " ".join(s.split())
    return s

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def resolve_revision(api: HfApi, repo_id: str, requested_revision: str) -> Tuple[str, Optional[str]]:
    """
    Resolve requested revision to a specific commit sha if possible.
    Returns (resolved_revision, resolved_sha) where resolved_revision is what we will pass to snapshot_download.
    """
    req = (requested_revision or "auto").strip()

    # If user already provided a commit-like sha, use it directly
    if len(req) >= 8 and all(c in "0123456789abcdef" for c in req.lower()) and len(req) <= 64:
        return req, req

    # auto / latest / main -> resolve to sha
    if req in ("auto", "latest", "main", "master", ""):
        try:
            info = api.model_info(repo_id)
            sha = getattr(info, "sha", None)
            if sha:
                return sha, sha
        except Exception as e:
            print(f"⚠️ Could not resolve sha for {repo_id} (requested={req}): {e}")
            # fallback: use "main" (not pinned)
            return "main", None

    # tag or branch name -> try resolve
    try:
        info = api.model_info(repo_id, revision=req)
        sha = getattr(info, "sha", None)
        if sha:
            return sha, sha
    except Exception as e:
        print(f"⚠️ Could not resolve sha for {repo_id} (revision={req}): {e}")

    # fallback to user input
    return req, None

def get_license_if_possible(api: HfApi, repo_id: str, revision: str) -> Optional[str]:
    try:
        info = api.model_info(repo_id, revision=revision)
        card = getattr(info, "cardData", None) or {}
        lic = card.get("license")
        return lic
    except Exception:
        return None

def download_snapshot(repo_id: str, revision: str, local_dir: Path) -> Path:
    """
    Download model snapshot into local_dir (no symlinks).
    Skip if local_dir already looks complete.
    """
    local_dir.mkdir(parents=True, exist_ok=True)

    # ✅ hard skip: already downloaded
    has_config = (local_dir / "config.json").exists()
    has_weights = any(local_dir.glob("*.safetensors")) or (local_dir / "pytorch_model.bin").exists()
    if has_config and has_weights:
        print(f"✅ Skip (already exists): {local_dir}")
        return local_dir

    print(f"\n⏳ Downloading: {repo_id}")
    print("   revision =", revision)
    print("   local_dir =", local_dir)

    snapshot_download(
        repo_id=repo_id,
        revision=revision,
        local_dir=str(local_dir),
        local_dir_use_symlinks=False,  # Windows friendly
        resume_download=True,
    )
    return local_dir


def build_manifest_header() -> Dict[str, Any]:
    return {
        "generated_at": now_iso(),
        "python_version": sys.version,
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "huggingface_hub_version": huggingface_hub.__version__,
        "hf_endpoint": os.environ.get("HF_ENDPOINT", ""),
    }

def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def write_yaml(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(obj, f, sort_keys=False, allow_unicode=True)


In [5]:
# Cell 5
api = HfApi()

FORCE_REDOWNLOAD = False
MAX_RETRIES = 5
SLEEP_BASE = 3  # seconds

def looks_downloaded(local_dir: Path) -> bool:
    """
    Heuristic: treat as downloaded if local_dir has config + weights/tokenizer-ish files.
    Avoids re-downloading on reruns.
    """
    if not local_dir.exists():
        return False
    # has any files?
    if not any(local_dir.rglob("*")):
        return False

    has_config = (local_dir / "config.json").exists()
    has_weights = bool(list(local_dir.glob("*.safetensors"))) or (local_dir / "pytorch_model.bin").exists()
    has_tokenizer = (local_dir / "tokenizer.json").exists() or (local_dir / "tokenizer_config.json").exists() or (local_dir / "vocab.json").exists()

    # Seq2Seq sometimes stores weights as *.safetensors; SentenceTransformer also has config.json-ish.
    return (has_config and (has_weights or has_tokenizer)) or has_weights

def download_with_retries(repo_id: str, revision: str, local_dir: Path) -> None:
    last_err = None
    for i in range(1, MAX_RETRIES + 1):
        try:
            print(f"\n⏳ Download attempt {i}/{MAX_RETRIES}: {repo_id} @ {revision}")
            snapshot_download(
                repo_id=repo_id,
                revision=revision,
                local_dir=str(local_dir),
                local_dir_use_symlinks=False,
                resume_download=True,
            )
            return
        except Exception as e:
            last_err = e
            wait = SLEEP_BASE * i
            print(f"⚠️ Download failed: {type(e).__name__}: {e}")
            print(f"   retry in {wait}s ...")
            time.sleep(wait)
    raise RuntimeError(f"Download failed after {MAX_RETRIES} retries: {repo_id}") from last_err


resolved_models = []
lock_registry = {
    "hf_endpoint": REGISTRY.get("hf_endpoint", ""),
    "runtime_profile": runtime_profile,
    "include_optional_vad": include_optional_vad,
    "models": [],
}

models_list = REGISTRY.get("models", [])

for m in models_list:
    model_key = m["model_key"]
    task = (m.get("task") or "").strip()

    if task == "vad" and not include_optional_vad:
        print(f"Skipping optional model (vad): {model_key}")
        continue

    repo_id = m["repo_id"]
    requested_revision = str(m.get("revision", "auto")).strip()
    local_dir_rel = m.get("local_dir", f"models/{model_key}")
    local_dir = (PROJECT_ROOT / local_dir_rel).resolve()

    # If optional force redownload
    if FORCE_REDOWNLOAD and local_dir.exists():
        import shutil
        shutil.rmtree(local_dir)

    # IMPORTANT: avoid extra network call before download
    # If requested_revision is "auto", use "main" for initial download
    revision_for_download = "main" if requested_revision in ("auto", "latest", "", None) else requested_revision

    # ✅ Skip if already downloaded (prevents re-download on reruns)
    if looks_downloaded(local_dir):
        print(f"✅ Skip download (already exists): {model_key} @ {local_dir}")
    else:
        local_dir.mkdir(parents=True, exist_ok=True)
        download_with_retries(repo_id, revision_for_download, local_dir)

    # Try to resolve sha AFTER (download or skip) (best-effort)
    resolved_sha = None
    license_str = None
    try:
        info = api.model_info(repo_id, revision=revision_for_download)
        resolved_sha = getattr(info, "sha", None)
        card = getattr(info, "cardData", None) or {}
        license_str = card.get("license")
    except Exception as e:
        print(f"⚠️ Could not fetch model_info for manifest (best-effort). {repo_id}: {e}")

    resolved_revision = resolved_sha or revision_for_download

    resolved_models.append({
        "model_key": model_key,
        "task": task,
        "repo_id": repo_id,
        "requested_revision": requested_revision,
        "revision": resolved_revision,
        "resolved_sha": resolved_sha,
        "license": license_str,
        "local_path": str(local_dir),
        "downloaded_at": now_iso(),
    })

    lock_registry["models"].append({
        "model_key": model_key,
        "task": task,
        "repo_id": repo_id,
        "revision": resolved_revision,   # if sha fetched, it's pinned; else 'main'
        "local_dir": local_dir_rel,
    })

print("\n✅ Download step completed.")


✅ Skip download (already exists): sentiment5_roberta_topic @ D:\coding\projects\Python\Youtube Comment Sentiment Analysis\models\sentiment5_roberta_topic
✅ Skip download (already exists): embedding_all-MiniLM-L6-v2 @ D:\coding\projects\Python\Youtube Comment Sentiment Analysis\models\embedding_all-MiniLM-L6-v2
✅ Skip download (already exists): summarizer_distilbart-cnn-12-6 @ D:\coding\projects\Python\Youtube Comment Sentiment Analysis\models\summarizer_distilbart-cnn-12-6
Skipping optional model (vad): vad_bert_optional

✅ Download step completed.


In [6]:
# Cell 6 — 写入 manifest + lockfile（可审计/可复现的证据资产）
# Optional: compute sha256 of key files for extra audit (costs time). Keep False by default.
COMPUTE_FILE_HASH = False

def list_key_files(model_dir: Path) -> List[str]:
    """
    Minimal key files we care about for integrity.
    Not all repos have same filenames, so we check existence.
    """
    candidates = [
        "config.json",
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "pytorch_model.bin",
        "model.safetensors",
        "sentence_bert_config.json",
        "modules.json",
    ]
    existing = []
    for c in candidates:
        p = model_dir / c
        if p.exists():
            existing.append(c)
    return existing

manifest = build_manifest_header()
manifest["models"] = []

for entry in resolved_models:
    model_dir = Path(entry["local_path"])
    files = list_key_files(model_dir)
    file_hashes = None
    if COMPUTE_FILE_HASH:
        file_hashes = {}
        for fn in files:
            file_hashes[fn] = sha256_file(model_dir / fn)

    manifest["models"].append({
        **entry,
        "key_files": files,
        "key_file_sha256": file_hashes,
    })

write_json(MANIFEST_PATH, manifest)
write_yaml(LOCK_PATH, lock_registry)

print("✅ Wrote manifest:", MANIFEST_PATH)
print("✅ Wrote lockfile:", LOCK_PATH)


✅ Wrote manifest: D:\coding\projects\Python\Youtube Comment Sentiment Analysis\models\manifest.json
✅ Wrote lockfile: D:\coding\projects\Python\Youtube Comment Sentiment Analysis\configs\model_registry.lock.yaml


In [7]:
# Cell 7 — Smoke Tests（含离线验证）
import numpy as np

def entropy_from_probs(probs: torch.Tensor, eps: float = 1e-12) -> float:
    p = probs + eps 
    p = p / p.sum()
    return float(-(p * torch.log(p)).sum().item())

def get_device(profile: str) -> torch.device:
    profile = (profile or "cpu_int8").strip().lower()
    if profile == "xpu":
        if hasattr(torch, "xpu") and torch.xpu.is_available():
            return torch.device("xpu")
        print("⚠️ XPU profile requested but torch.xpu is not available. Falling back to CPU.")
    return torch.device("cpu")

DEVICE = get_device(runtime_profile)
print("Using DEVICE =", DEVICE)

def smoke_sentiment5(model_dir: Path, target: str = "SHEIN") -> Dict[str, Any]:
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True)
    model.to(DEVICE)
    model.eval()
    load_ms = int((time.time() - t0) * 1000)

    id2label = getattr(model.config, "id2label", {}) or {}
    label_to_weight = {
        "strongly negative": -2,
        "negative": -1,
        "negative or neutral": 0,
        "neutral": 0,
        "positive": 1,
        "strongly positive": 2,
    }
    weights = []
    for i in range(model.config.num_labels):
        lab = normalize_label(id2label.get(i, f"label_{i}"))
        if lab in label_to_weight:
            weights.append(label_to_weight[lab])
        else:
            fallback = [-2, -1, 0, 1, 2]
            weights.append(fallback[i] if i < len(fallback) else 0)

    sample_comment = "I can't believe this brand treats workers like this. This is unacceptable."
    text_input = f"{sample_comment} </s> {target}"

    t1 = time.time()
    with torch.no_grad():
        inputs = tok(text_input, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        logits = model(**inputs).logits[0]
        probs = torch.softmax(logits, dim=-1).detach().cpu()

    infer_ms = int((time.time() - t1) * 1000)

    pred_id = int(torch.argmax(probs).item())
    pred_label = id2label.get(pred_id, str(pred_id))

    expected_valence = float(sum(float(probs[i]) * weights[i] for i in range(len(weights))))
    extremity = abs(expected_valence)
    uncertainty = entropy_from_probs(probs)

    return {
        "load_ok": True,
        "inference_ok": True,
        "device": str(DEVICE),
        "num_labels": model.config.num_labels,
        "id2label": {str(k): v for k, v in id2label.items()},
        "sent5_target_used": target,
        "sample_pred_label": pred_label,
        "sample_probs": [float(x) for x in probs.tolist()],
        "sent5_valence_expected": expected_valence,
        "sent5_extremity": extremity,
        "sent5_uncertainty_entropy": uncertainty,
        "load_ms": load_ms,
        "infer_ms": infer_ms,
    }

def smoke_embedding(model_dir: Path) -> Dict[str, Any]:
    t0 = time.time()
    model = SentenceTransformer(str(model_dir))
    load_ms = int((time.time() - t0) * 1000)

    texts = [
        "This is a fast fashion brand.",
        "These comments discuss labor exploitation and worker rights.",
    ]
    t1 = time.time()
    emb = model.encode(texts, normalize_embeddings=True)
    infer_ms = int((time.time() - t1) * 1000)

    emb = np.asarray(emb)
    dim = int(emb.shape[1])
    cosine = float((emb[0] @ emb[1]).item())

    return {
        "load_ok": True,
        "inference_ok": True,
        "embedding_dim": dim,
        "sample_cosine_similarity": cosine,
        "load_ms": load_ms,
        "infer_ms": infer_ms,
    }

def smoke_summarizer(model_dir: Path) -> Dict[str, Any]:
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_dir, local_files_only=True)
    model.to(DEVICE)
    model.eval()
    load_ms = int((time.time() - t0) * 1000)

    text = (
        "People in the comments argue about whether the brand uses forced labor, "
        "and many users express anger and call for boycotts. Others dispute the claims."
    )
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)

    t1 = time.time()
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=2,
            do_sample=False,
        )
    summary = tok.decode(out_ids[0], skip_special_tokens=True)
    infer_ms = int((time.time() - t1) * 1000)

    ok = bool(summary.strip())
    return {
        "load_ok": True,
        "inference_ok": ok,
        "sample_summary": summary,
        "load_ms": load_ms,
        "infer_ms": infer_ms,
    }

def smoke_vad(model_dir: Path) -> Dict[str, Any]:
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True)
    model.to(DEVICE)
    model.eval()
    load_ms = int((time.time() - t0) * 1000)

    text = "I feel extremely angry about these working conditions."
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)

    t1 = time.time()
    with torch.no_grad():
        logits = model(**inputs).logits[0].detach().cpu().numpy()
    infer_ms = int((time.time() - t1) * 1000)

    return {
        "load_ok": True,
        "inference_ok": True,
        "raw_output": [float(x) for x in logits.tolist()],
        "load_ms": load_ms,
        "infer_ms": infer_ms,
    }

# ✅ NEW: NLI smoke test
def smoke_nli(model_dir: Path) -> Dict[str, Any]:
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True)
    model.to(DEVICE)
    model.eval()
    load_ms = int((time.time() - t0) * 1000)

    premise = "I like this product."
    hypo_support = "This product is great."
    hypo_oppose = "This product is terrible."

    t1 = time.time()
    with torch.no_grad():
        batch = tok(
            [premise, premise],
            [hypo_support, hypo_oppose],
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256,
        )
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch).logits
        probs = torch.softmax(logits, dim=-1).detach().cpu()
    infer_ms = int((time.time() - t1) * 1000)

    pred = probs.argmax(dim=-1).tolist()
    labels = [model.config.id2label.get(i, str(i)) for i in pred]

    return {
        "load_ok": True,
        "inference_ok": True,
        "sample_probs": probs.tolist(),
        "sample_labels": labels,
        "load_ms": load_ms,
        "infer_ms": infer_ms,
    }

# --------------------------
# Run smoke tests (online/local)
# --------------------------
smoke = {
    "generated_at": now_iso(),
    "profile": runtime_profile,
    "device": str(DEVICE),
    "hf_endpoint": os.environ.get("HF_ENDPOINT", ""),
    "tests": {},
}

# locate model dirs from lock_registry (preferred) or registry
model_dirs = {m["model_key"]: (PROJECT_ROOT / m["local_dir"]).resolve() for m in lock_registry["models"]}

for m in lock_registry["models"]:
    key = m["model_key"]
    task = m["task"]
    path = model_dirs[key]
    print(f"\n🔎 Smoke testing: {key} ({task}) @ {path}")

    try:
        if task == "sentiment5":
            smoke["tests"][key] = smoke_sentiment5(path, target="SHEIN")
        elif task == "embedding":
            smoke["tests"][key] = smoke_embedding(path)
        elif task == "summarization":
            smoke["tests"][key] = smoke_summarizer(path)
        elif task == "vad":
            smoke["tests"][key] = smoke_vad(path)
        elif task == "nli":
            smoke["tests"][key] = smoke_nli(path)
        else:
            smoke["tests"][key] = {"load_ok": False, "inference_ok": False, "error": f"Unknown task: {task}"}
    except Exception as e:
        smoke["tests"][key] = {"load_ok": False, "inference_ok": False, "error": str(e)}

# --------------------------
# Offline smoke tests (critical)
# --------------------------
def with_offline_env(fn):
    old_env = dict(os.environ)
    try:
        os.environ["TRANSFORMERS_OFFLINE"] = "1"
        os.environ["HF_HUB_OFFLINE"] = "1"
        return fn()
    finally:
        os.environ.clear()
        os.environ.update(old_env)

def run_offline_tests() -> Dict[str, Any]:
    offline = {"tests": {}}
    for m in lock_registry["models"]:
        key = m["model_key"]
        task = m["task"]
        path = model_dirs[key]
        try:
            if task == "sentiment5":
                offline["tests"][key] = smoke_sentiment5(path, target="SHEIN")
            elif task == "embedding":
                offline["tests"][key] = smoke_embedding(path)
            elif task == "summarization":
                offline["tests"][key] = smoke_summarizer(path)
            elif task == "vad":
                offline["tests"][key] = smoke_vad(path)
            elif task == "nli":
                offline["tests"][key] = smoke_nli(path)
        except Exception as e:
            offline["tests"][key] = {"load_ok": False, "inference_ok": False, "error": str(e)}
    return offline

print("\n🚫 Running OFFLINE smoke tests ...")
offline_result = with_offline_env(run_offline_tests)
smoke["offline"] = offline_result

# write report
write_json(SMOKE_REPORT_PATH, smoke)
print("✅ Wrote smoke test report:", SMOKE_REPORT_PATH)

# quick summary
failed = []
for k, r in smoke["tests"].items():
    if not (r.get("load_ok") and r.get("inference_ok")):
        failed.append(k)
offline_failed = []
for k, r in smoke["offline"]["tests"].items():
    if not (r.get("load_ok") and r.get("inference_ok")):
        offline_failed.append(k)

print("\n==== Summary ====")
print("Failed (online/local):", failed or "None")
print("Failed (offline):", offline_failed or "None")

if offline_failed:
    raise RuntimeError(f"PIPELINE BLOCKED: Offline smoke tests failed for {offline_failed}")
else:
    print("✅ PIPELINE OK: Offline-ready confirmed.")


Using DEVICE = cpu

🔎 Smoke testing: sentiment5_roberta_topic (sentiment5) @ D:\coding\projects\Python\Youtube Comment Sentiment Analysis\models\sentiment5_roberta_topic

🔎 Smoke testing: embedding_all-MiniLM-L6-v2 (embedding) @ D:\coding\projects\Python\Youtube Comment Sentiment Analysis\models\embedding_all-MiniLM-L6-v2

🔎 Smoke testing: summarizer_distilbart-cnn-12-6 (summarization) @ D:\coding\projects\Python\Youtube Comment Sentiment Analysis\models\summarizer_distilbart-cnn-12-6


c:\Users\76724\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\generation\utils.py:1633: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (33). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length. Note that `max_length` is set to 33, its default value.
  warnings.warn(



🚫 Running OFFLINE smoke tests ...
✅ Wrote smoke test report: D:\coding\projects\Python\Youtube Comment Sentiment Analysis\reports\00_smoke_test.json

==== Summary ====
Failed (online/local): None
Failed (offline): None
✅ PIPELINE OK: Offline-ready confirmed.
